# model-train-eval-toggle-around-sample — worked example 3: Toggle preserves a model already in eval

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `model-train-eval-toggle-around-sample`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import torch.nn as nn
import matplotlib.pyplot as plt

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

A robust eval wrapper must not force `train()` on exit if the model was already in eval mode. By saving `was_training` and only restoring `train()` when it was True, a model that entered in eval stays in eval afterward.

## Worked solution

We verify the wrapper respects the model's incoming mode.

1. **Start in eval.** We call `model.eval()` before using the context manager.
2. **Enter the wrapper.** It records `was_training = False`, sets eval (already eval), and yields under no_grad.
3. **finally.** Because `was_training` is False, the `finally` does NOT call `train()`; the model stays in eval.
4. **Contrast.** We also run it on a model that started in train and confirm that one is restored to train.

The demo prints the post-block training flag for both an eval-start and a train-start model, showing each is left as it began.

In [ ]:
import torch as t
import torch.nn as nn
import contextlib

t.manual_seed(2)

@contextlib.contextmanager
def eval_mode(model):
    was_training = model.training
    model.eval()
    try:
        with t.no_grad():
            yield model
    finally:
        if was_training:
            model.train()

m_eval = nn.Linear(2, 2).eval()
with eval_mode(m_eval):
    pass
print('eval-start stays eval:', not m_eval.training)

m_train = nn.Linear(2, 2).train()
with eval_mode(m_train):
    pass
print('train-start restored to train:', m_train.training)